In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.pylab import rcParams
rcParams['figure.figsize'] = (20, 10)
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout

In [3]:
data = pd.read_csv("KAYNES.csv")
data.head()

,Symbol,Series,Date,Prev Close,Open Price,High Price,Low Price,Last Price,Close Price,Average Price,Total Traded Quantity,Turnover,No. of Trades
0,KAYNES,EQ,27-Feb-26,"3,916.40","3,916.40","3,958.10","3,822.10","3,845.00","3,856.50","3,874.40","10,25,078","3,97,15,65,854.90","57,088"
1,KAYNES,EQ,26-Feb-26,"3,908.40","3,916.00","3,990.60","3,878.00","3,919.00","3,916.40","3,931.94","9,86,604","3,87,92,65,303.10","53,195"
2,KAYNES,EQ,25-Feb-26,"3,838.60","3,871.00","3,982.00","3,865.10","3,916.20","3,908.40","3,919.71","13,05,141","5,11,57,78,260.00","68,429"
3,KAYNES,EQ,24-Feb-26,"3,799.50","3,785.00","3,868.00","3,745.90","3,860.00","3,838.60","3,794.64","8,93,784","3,39,15,85,282.00","50,485"
4,KAYNES,EQ,23-Feb-26,"3,870.90","3,898.00","3,899.50","3,785.00","3,811.90","3,799.50","3,829.13","9,85,892","3,77,51,11,790.00","52,717"


In [4]:
print(data.columns.tolist())

['Symbol  ', 'Series  ', 'Date  ', 'Prev Close  ', 'Open Price  ', 'High Price  ', 'Low Price  ', 'Last Price  ', 'Close Price  ', 'Average Price ', 'Total Traded Quantity  ', 'Turnover', 'No. of Trades  ']


In [5]:
## Data Cleaning
# Dropping unwanted columns

data.columns = data.columns.str.strip() # removing white spaces so that we don't get any key errors

data = data.drop(columns=["Symbol", "Series", "Prev Close", "Average Price", "No. of Trades"], errors="ignore")
data.head()

,Date,Open Price,High Price,Low Price,Last Price,Close Price,Total Traded Quantity,Turnover
0,27-Feb-26,"3,916.40","3,958.10","3,822.10","3,845.00","3,856.50","10,25,078","3,97,15,65,854.90"
1,26-Feb-26,"3,916.00","3,990.60","3,878.00","3,919.00","3,916.40","9,86,604","3,87,92,65,303.10"
2,25-Feb-26,"3,871.00","3,982.00","3,865.10","3,916.20","3,908.40","13,05,141","5,11,57,78,260.00"
3,24-Feb-26,"3,785.00","3,868.00","3,745.90","3,860.00","3,838.60","8,93,784","3,39,15,85,282.00"
4,23-Feb-26,"3,898.00","3,899.50","3,785.00","3,811.90","3,799.50","9,85,892","3,77,51,11,790.00"


In [6]:
print(data.columns.tolist())

['Date', 'Open Price', 'High Price', 'Low Price', 'Last Price', 'Close Price', 'Total Traded Quantity', 'Turnover']


In [7]:
# renaming columns in DataFrame

df = data.rename(columns={
    "Date": "Date",
    "Open Price": "Open",
    "High Price": "High",
    "Low Price": "Low",
    "Last Price": "Last",
    "Close Price": "Close",
    "Total Traded Quantity": "Volume",
    "Turnover": "Turnover"
})


In [8]:
# lowercasing the dataframe
df.columns = df.columns.str.strip().str.lower()
df.columns.tolist()

['date', 'open', 'high', 'low', 'last', 'close', 'volume', 'turnover']

In [9]:
# Remove commas and convert them just to numeric columns except date

cols = ['open', 'high', 'low', 'last', 'close', 'volume', 'turnover']

for col in cols:
    df[col] = df[col].astype(str).str.replace(',', '', regex=False)
    df[col] = pd.to_numeric(df[col], errors='coerce')


In [10]:
df.head()

,date,open,high,low,last,close,volume,turnover
0,27-Feb-26,3916.4,3958.1,3822.1,3845.0,3856.5,1025078,3.971566e+09
1,26-Feb-26,3916.0,3990.6,3878.0,3919.0,3916.4,986604,3.879265e+09
2,25-Feb-26,3871.0,3982.0,3865.1,3916.2,3908.4,1305141,5.115778e+09
3,24-Feb-26,3785.0,3868.0,3745.9,3860.0,3838.6,893784,3.391585e+09
4,23-Feb-26,3898.0,3899.5,3785.0,3811.9,3799.5,985892,3.775112e+09


In [11]:
# since during the numeric conversion the turnover column took exponential values converting them back to simple integers
df['turnover'] = df['turnover'].astype('int64')
df.head()

,date,open,high,low,last,close,volume,turnover
0,27-Feb-26,3916.4,3958.1,3822.1,3845.0,3856.5,1025078,3971565854
1,26-Feb-26,3916.0,3990.6,3878.0,3919.0,3916.4,986604,3879265303
2,25-Feb-26,3871.0,3982.0,3865.1,3916.2,3908.4,1305141,5115778260
3,24-Feb-26,3785.0,3868.0,3745.9,3860.0,3838.6,893784,3391585282
4,23-Feb-26,3898.0,3899.5,3785.0,3811.9,3799.5,985892,3775111790


In [11]:
## formatting  date column and setting it as index - important step

df['date'] = pd.to_datetime(df['date'], format='%d-%b-%y')


In [12]:
df = df.sort_values('date')
df.set_index('date', inplace=True)
df.head()

,open,high,low,last,close,volume,turnover
date,,,,,,,
2022-11-28,745.00,764.00,734.55,750.0,749.90,759905,569953081
2022-11-29,753.70,754.75,731.15,733.0,733.95,462934,343129859
2022-11-30,735.00,738.00,722.15,725.5,725.15,347873,252570376
2022-12-01,728.25,745.25,728.25,738.0,739.45,279131,206111097
2022-12-02,738.95,747.95,724.00,741.0,740.20,328645,242418798


In [13]:
df = df.dropna()

In [14]:
# checking for nul values and datatypes
df.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 810 entries, 2022-11-28 to 2026-02-27
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   open      810 non-null    float64
 1   high      810 non-null    float64
 2   low       810 non-null    float64
 3   last      810 non-null    float64
 4   close     810 non-null    float64
 5   volume    810 non-null    int64  
 6   turnover  810 non-null    int64  
dtypes: float64(5), int64(2)
memory usage: 50.6 KB


In [15]:
# plotting the chart to replicate the price movements on a line chart using just the closing price
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=df.index,
    y=df['close'],
    mode='lines',
    name='Close Price'
))

fig.update_layout(
    title="Closing Price Chart",
    xaxis_title="Date",
    yaxis_title="Price",
    hovermode="x unified"
)

fig.show()

In [16]:
# creating a function to profile volume - how much trading volume happened at different price levels
def volume_profile(df, price_col='close', volume_col='volume', bins=50):
    price_min = df[price_col].min()
    price_max = df[price_col].max()

    # Create price bins
    bins_edges = np.linspace(price_min, price_max, bins)

    # Assign each row to a bin
    df['price_bin'] = pd.cut(df[price_col], bins=bins_edges)

    # Aggregate volume per bin
    vp = df.groupby('price_bin', observed=True)[volume_col].sum()

    return vp

vp = volume_profile(df)
print(vp)

price_bin
(680.25, 821.419]       10586252
(821.419, 962.589]       8533829
(962.589, 1103.758]      3178820
(1103.758, 1244.928]     3840184
(1244.928, 1386.097]     2240953
(1386.097, 1527.266]     2131206
(1527.266, 1668.436]     4984145
(1668.436, 1809.605]     4878363
(1809.605, 1950.774]     1932834
(1950.774, 2091.944]     4662868
(2091.944, 2233.113]     1219527
(2233.113, 2374.283]     2057647
(2374.283, 2515.452]     4339926
(2515.452, 2656.621]     7152778
(2656.621, 2797.791]     7287236
(2797.791, 2938.96]      6391065
(2938.96, 3080.13]        985444
(3080.13, 3221.299]      3252308
(3221.299, 3362.468]     2746672
(3362.468, 3503.638]    12602696
(3503.638, 3644.807]    10000362
(3644.807, 3785.977]    20393280
(3785.977, 3927.146]    40362271
(3927.146, 4068.315]    37486221
(4068.315, 4209.485]    30918764
(4209.485, 4350.654]    37193425
(4350.654, 4491.823]    16340719
(4491.823, 4632.993]     4077142
(4632.993, 4774.162]    10816347
(4774.162, 4915.332]    10728389


In [17]:
# Convert index (price bins) to string for clean display
price_ranges = vp.index.astype(str)

fig = go.Figure()

fig.add_trace(go.Bar(
    x=vp.values,
    y=price_ranges,
    orientation='h',
    name='Volume Profile'
))

fig.update_layout(
    title="Volume Profile - Volume vs Price ", # - At which price levels did the most trading happen
    xaxis_title="Volume",
    yaxis_title="Price Range",
    hovermode="y"
)

fig.show()

In [18]:
# stacking price at the top and volume at the bottom to give a better understanding between the volume and price relationship

fig = make_subplots(
    rows=2, cols=1,
    shared_xaxes=True,
    vertical_spacing=0.05,
    row_heights=[0.7, 0.3]
)

# --- Price ---
fig.add_trace(
    go.Scatter(
        x=df.index,
        y=df['close'],
        mode='lines',
        name='Close Price',
        line=dict(color='blue', width=2)
    ),
    row=1, col=1
)

# --- Volume ---
colors = ['green' if df['close'].iloc[i] >= df['close'].iloc[i-1] else 'red'
          for i in range(1, len(df))]

fig.add_trace(
    go.Bar(
        x=df.index[1:],
        y=df['volume'][1:],
        marker=dict(
            color=colors,
            line=dict(color='black', width=0.5)
        ),
        opacity=0.9,
        name='Volume'
    ),
    row=2, col=1
)

# --- Layout ---
fig.update_layout(
    title="Price + Volume",
    plot_bgcolor='#f5f5f5',
    paper_bgcolor='#f5f5f5',
    font=dict(color='black'),
    hovermode='x unified',
    height=600
)

# Range slider ONLY for top axis
fig.update_layout(
    xaxis=dict(rangeslider=dict(visible=True))
)

fig.show()

In [19]:
# we will be focusing on closing price and volume with date column as index
features = df[['close', 'volume']]
features.head()

,close,volume
date,,
2022-11-28,749.90,759905
2022-11-29,733.95,462934
2022-11-30,725.15,347873
2022-12-01,739.45,279131
2022-12-02,740.20,328645


In [20]:
dataset_new = df[['close', 'volume']].copy()
dataset_new.describe()

,close,volume
count,810.000000,8.100000e+02
mean,3736.610370,5.836861e+05
std,1987.893923,1.192189e+06
min,680.250000,1.134500e+04
25%,2118.650000,1.549752e+05
50%,3784.700000,2.883125e+05
75%,5597.000000,5.863570e+05
max,7597.550000,1.844877e+07


In [21]:
# feature engineering - using 10 and 20 ema and 10 Ema
df = dataset_new.copy() # creating a copy to avoid meddling with original dataset

df['ma10'] = df['close'].rolling(10).mean() # Smooths noise  -  shows short-term trend
df['ma20'] = df['close'].rolling(20).mean()  # Used to detect trend direction and crossovers
df['ema10'] = df['close'].ewm(span=10).mean() # Gives more weight to recent prices - Reacts faster to changes
df['returns'] = df['close'].pct_change() # calculates percentage change from previous price
df['vol_change'] = df['volume'].pct_change() # measurs trading aactivity changes

df = df.dropna()

In [22]:
features = ['close', 'volume', 'ma10', 'ma20', 'ema10', 'returns', 'vol_change']
data = df[features]
data.head()

,close,volume,ma10,ma20,ema10,returns,vol_change
date,,,,,,,
2022-12-23,680.25,177531,709.935,725.1350,706.530184,-0.031328,-0.115441
2022-12-26,703.85,133235,707.240,722.8325,706.035564,0.034693,-0.249511
2022-12-27,721.70,123539,706.810,722.2200,708.918520,0.025361,-0.072774
2022-12-28,737.75,196026,707.100,722.8500,714.213012,0.022239,0.586754
2022-12-29,747.45,172325,709.650,723.2500,720.305439,0.013148,-0.120907


In [23]:
# scaling per feature - using min-max scaler, confining the values between 0-1

scalers = {}
scaled_cols = []

for col in features:
    scaler = MinMaxScaler()
    scaled = scaler.fit_transform(data[[col]])

    scalers[col] = scaler
    scaled_cols.append(scaled)

scaled_data = np.hstack(scaled_cols)

In [24]:
print(f'Scalers: {scalers}')
print(f'Scaled data: {scaled_data}')

Scalers: {'close': MinMaxScaler(), 'volume': MinMaxScaler(), 'ma10': MinMaxScaler(), 'ma20': MinMaxScaler(), 'ema10': MinMaxScaler(), 'returns': MinMaxScaler(), 'vol_change': MinMaxScaler()}
Scaled data: [[0.00000000e+00 9.01351414e-03 4.72151735e-04 ... 7.48092471e-05
  4.12346285e-01 2.47383760e-02]
 [3.41173579e-03 6.61100958e-03 6.49680788e-05 ... 0.00000000e+00
  5.80195556e-01 2.07863407e-02]
 [5.99222240e-03 6.08512273e-03 0.00000000e+00 ... 4.36035706e-04
  5.56468684e-01 2.59960931e-02]
 ...
 [4.66677750e-01 7.01722681e-02 4.86978055e-01 ... 4.80720857e-01
  5.38222651e-01 4.17079662e-02]
 [4.67834271e-01 5.28956157e-02 4.84406528e-01 ... 4.81599786e-01
  4.97196968e-01 2.09469351e-02]
 [4.59174823e-01 5.49823495e-02 4.83107166e-01 ... 4.80671702e-01
  4.53108521e-01 2.92907746e-02]]


In [25]:
print(scaled_data[:5])
print(scaled_data[-5:])

[[0.00000000e+00 9.01351414e-03 4.72151735e-04 4.47414932e-04
  7.48092471e-05 4.12346285e-01 2.47383760e-02]
 [3.41173579e-03 6.61100958e-03 6.49680788e-05 9.40108561e-05
  0.00000000e+00 5.80195556e-01 2.07863407e-02]
 [5.99222240e-03 6.08512273e-03 0.00000000e+00 0.00000000e+00
  4.36035706e-04 5.56468684e-01 2.59960931e-02]
 [8.31249187e-03 1.00166368e-02 4.38156810e-05 9.66968806e-05
  1.23680674e-03 5.48533059e-01 4.54371976e-02]
 [9.71477311e-03 8.73115369e-03 4.29091497e-04 1.58091725e-04
  2.15826235e-03 5.25420265e-01 2.45772390e-02]]
[[0.45093461 0.052857   0.49249732 0.469745   0.48127715 0.44509839
  0.01994066]
 [0.45658711 0.04786129 0.49069937 0.47241568 0.47991549 0.51815608
  0.02538732]
 [0.46667775 0.07017227 0.48697806 0.4762989  0.48072086 0.53822265
  0.04170797]
 [0.46783427 0.05289562 0.48440653 0.47968329 0.48159979 0.49719697
  0.02094694]
 [0.45917482 0.05498235 0.48310717 0.48195797 0.4806717  0.45310852
  0.02929077]]


In [26]:
print(scaled_data.min(), scaled_data.max())

0.0 1.0000000000000002


In [27]:
print(scaled_data[:3])
print(scaled_data[-3:])

[[0.00000000e+00 9.01351414e-03 4.72151735e-04 4.47414932e-04
  7.48092471e-05 4.12346285e-01 2.47383760e-02]
 [3.41173579e-03 6.61100958e-03 6.49680788e-05 9.40108561e-05
  0.00000000e+00 5.80195556e-01 2.07863407e-02]
 [5.99222240e-03 6.08512273e-03 0.00000000e+00 0.00000000e+00
  4.36035706e-04 5.56468684e-01 2.59960931e-02]]
[[0.46667775 0.07017227 0.48697806 0.4762989  0.48072086 0.53822265
  0.04170797]
 [0.46783427 0.05289562 0.48440653 0.47968329 0.48159979 0.49719697
  0.02094694]
 [0.45917482 0.05498235 0.48310717 0.48195797 0.4806717  0.45310852
  0.02929077]]


In [28]:
import joblib

joblib.dump(scalers, "scalers.pkl")

['scalers.pkl']

In [29]:
lookback = 60 # we use last 60 days data points
split_index = int(len(df) * 0.8)

X_train, y_train = [], []
X_test, y_test = [], []

for i in range(lookback, len(scaled_data)):

    X_seq = scaled_data[i-lookback:i]
    y_seq = scaled_data[i][:2]  # predict close + volume  - first 2 data points

    if i < split_index:
        X_train.append(X_seq)
        y_train.append(y_seq)
    else:
        X_test.append(X_seq)
        y_test.append(y_seq)
# converting them to array to feed into the model
X_train, y_train = np.array(X_train), np.array(y_train)
X_test, y_test = np.array(X_test), np.array(y_test)

In [30]:
print("Train last date:", dataset_new.index[split_index-1])
print("Test first date:", dataset_new.index[split_index])

Train last date: 2025-06-13 00:00:00
Test first date: 2025-06-16 00:00:00


In [31]:
print(len(X_test), len(dataset_new) - split_index)

159 178


In [32]:
print("Train last:", dataset_new.index[split_index-1])
print("Test first:", dataset_new.index[split_index])
print("X_test shape:", X_test.shape)

Train last: 2025-06-13 00:00:00
Test first: 2025-06-16 00:00:00
X_test shape: (159, 60, 7)


In [33]:
#early Stopping
# Stops training when validation loss stops improving and keeps the best model
early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)

In [80]:
# Defining the model as per the results of hyper-parameter tuning values
model = Sequential()

model.add(LSTM(128, return_sequences=True, input_shape=(lookback, len(features))))
model.add(Dropout(0.2))

model.add(LSTM(128))
model.add(Dropout(0.2))

model.add(Dense(32))
model.add(Dense(2))

model.compile(optimizer='adam', loss='mean_squared_error')
model.summary()

Model: "sequential_7"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 lstm_14 (LSTM)              (None, 60, 128)           69632     
                                                                 
 dropout_14 (Dropout)        (None, 60, 128)           0         
                                                                 
 lstm_15 (LSTM)              (None, 128)               131584    
                                                                 
 dropout_15 (Dropout)        (None, 128)               0         
                                                                 
 dense_14 (Dense)            (None, 32)                4128      
                                                                 
 dense_15 (Dense)            (None, 2)                 66        
                                                                 
Total params: 205410 (802.38 KB)
Trainable params: 205

In [81]:
history = model.fit(X_train, y_train, epochs=30, batch_size=32, callbacks=[early_stopping])

Epoch 1/30
18/18 [==============================] - 8s 107ms/step - loss: 0.0217
Epoch 2/30
18/18 [==============================] - 2s 101ms/step - loss: 0.0041
Epoch 3/30
18/18 [==============================] - 2s 106ms/step - loss: 0.0031
Epoch 4/30
18/18 [==============================] - 2s 108ms/step - loss: 0.0025
Epoch 5/30
18/18 [==============================] - 2s 103ms/step - loss: 0.0025
Epoch 6/30
18/18 [==============================] - 2s 103ms/step - loss: 0.0022
Epoch 7/30
18/18 [==============================] - 2s 106ms/step - loss: 0.0020
Epoch 8/30
18/18 [==============================] - 2s 106ms/step - loss: 0.0018
Epoch 9/30
18/18 [==============================] - 2s 114ms/step - loss: 0.0019
Epoch 10/30
18/18 [==============================] - 2s 113ms/step - loss: 0.0020
Epoch 11/30
18/18 [==============================] - 2s 116ms/step - loss: 0.0017
Epoch 12/30
18/18 [==============================] - 2s 101ms/step - loss: 0.0015
Epoch 13/30
18/18 [======

In [82]:
model.save("stock_prediction.h5")

C:\Users\DELL\PyCharmMiscProject\.venv\Lib\site-packages\keras\src\engine\training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


In [83]:
# predicting the model output using test data
predictions = model.predict(X_test)
#converts model outputs from scaled values to real stock price and volume using inverse transform
pred_close = scalers['close'].inverse_transform(predictions[:, 0].reshape(-1,1)).flatten()
pred_volume = scalers['volume'].inverse_transform(predictions[:, 1].reshape(-1,1)).flatten()

5/5 [==============================] - 4s 65ms/step


In [84]:
# aligning the predicted values with correct dates and stores them in a structured DataFrame.
pred_index = df.index[split_index:]
pred_index = pred_index[:len(pred_close)]

pred_df = pd.DataFrame({
    'Pred_close': pred_close,
    'Pred_volume': pred_volume
}, index=pred_index)

In [85]:
print(pred_df.head())
print(pred_df.tail())

             Pred_close   Pred_volume
date                                 
2025-07-11  6158.856445  451248.21875
2025-07-14  6123.708008  458356.68750
2025-07-15  6098.303711  461025.78125
2025-07-16  6062.142090  465232.56250
2025-07-17  6035.608887  472786.81250
             Pred_close   Pred_volume
date                                 
2026-02-23  3994.582764  478862.18750
2026-02-24  3951.989014  468630.34375
2026-02-25  3934.046631  452748.15625
2026-02-26  3945.690674  438659.75000
2026-02-27  3952.949219  425629.21875


In [86]:
# predicts future stock values step-by-step by repeatedly feeding previous predictions back into the model.
future_steps = 7
future_data = df.copy()
future_preds = []


# predicts 7 days a head by looping through future days
for _ in range(future_steps):

    # take last window
    recent = future_data.iloc[-lookback:]

    # scale window
    scaled_window = []
    for col in features:
        scaled = scalers[col].transform(recent[[col]])
        scaled_window.append(scaled)

    scaled_window = np.hstack(scaled_window)

    # predict
    pred = model.predict(scaled_window.reshape(1, lookback, len(features)))[0]

    # inverse scale
    pred_close = scalers['close'].inverse_transform([[pred[0]]])[0][0]
    pred_volume = scalers['volume'].inverse_transform([[pred[1]]])[0][0]

    # create new row - the next day row
    next_date = future_data.index[-1] + pd.Timedelta(days=1)

    new_row = pd.DataFrame({
        'close': [pred_close],
        'volume': [pred_volume]
    }, index=[next_date])

    # append and recompute features - Recalculates indicators using new predicted data
    temp = pd.concat([future_data, new_row])

    temp['ma10'] = temp['close'].rolling(10).mean()
    temp['ma20'] = temp['close'].rolling(20).mean()
    temp['ema10'] = temp['close'].ewm(span=10).mean()
    temp['returns'] = temp['close'].pct_change()
    temp['vol_change'] = temp['volume'].pct_change()

    temp = temp.dropna()

    future_data = temp.copy()

    future_preds.append([pred_close, pred_volume])

1/1 [==============================] - 0s 60ms/step


In [89]:
future_preds

array([[  3938.36540812, 419944.377608  ],
       [  3947.55317284, 402033.8294121 ],
       [  3953.79400036, 385356.55049991],
       [  3958.19389406, 370222.78995134],
       [  3961.4490279 , 356832.6951693 ],
       [  3964.2050687 , 345218.61667773],
       [  3967.29672432, 335136.03671605]])

In [87]:
#converting predicted values into a properly indexed DataFrame with future dates.
future_preds = np.array(future_preds)

future_dates = pd.date_range(
    start=df.index[-1] + pd.Timedelta(days=1),
    periods=future_steps
)

future_df = pd.DataFrame({
    'close': future_preds[:, 0],
    'volume': future_preds[:, 1]
}, index=future_dates)

In [90]:
future_df

,close,volume
2026-02-28,3938.365408,419944.377608
2026-03-01,3947.553173,402033.829412
2026-03-02,3953.794000,385356.550500
2026-03-03,3958.193894,370222.789951
2026-03-04,3961.449028,356832.695169
2026-03-05,3964.205069,345218.616678
2026-03-06,3967.296724,335136.036716


In [88]:
#uilding a 2-layer interactive chart showing actual, predicted, and future stock price & volume with highlighted prediction zones.

fig = make_subplots(
    rows=2, cols=1,
    shared_xaxes=True,
    vertical_spacing=0.05,
    row_heights=[0.7, 0.3]
)

# --- PRICE ---
fig.add_trace(go.Scatter(
    x=df.index,
    y=df['close'],
    mode='lines',
    name='Actual',
    line=dict(color='blue')
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=pred_df.index,
    y=pred_df['Pred_close'],
    mode='lines',
    name='Predicted',
    line=dict(color='red')
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=future_df.index,
    y=future_df['close'],
    mode='lines+markers',
    name='Future 7 Days',
    line=dict(color='orange', dash='dash')
), row=1, col=1)

# --- Highlight prediction region ---
fig.add_vrect(
    x0=pred_df.index[0],
    x1=future_df.index[-1],
    fillcolor="rgba(255, 0, 0, 0.1)",
    line_width=0
)

# --- VOLUME ---
colors = ['green' if df['close'].iloc[i] >= df['close'].iloc[i-1] else 'red'
          for i in range(1, len(df))]

fig.add_trace(go.Bar(
    x=df.index[1:],
    y=df['volume'][1:],
    marker_color=colors,
    name='Actual Volume'
), row=2, col=1)

fig.add_trace(go.Scatter(
    x=pred_df.index,
    y=pred_df['Pred_volume'],
    mode='lines',
    name='Predicted Volume',
    line=dict(color='red')
), row=2, col=1)

fig.add_trace(go.Scatter(
    x=future_df.index,
    y=future_df['volume'],
    mode='lines+markers',
    name='Future Volume',
    line=dict(color='orange', dash='dash')
), row=2, col=1)

# --- Layout ---
fig.update_layout(
    title="Price & Volume Prediction",
    hovermode='x unified',
    plot_bgcolor='#f5f5f5',
    paper_bgcolor='#f5f5f5',
    font=dict(color='black'),
    xaxis_rangeslider_visible=True,
    height=700
)

fig.show()

### Hyper Parameter tunning



In [43]:
from tensorflow.keras.optimizers import Adam

def build_model(units=64, dropout=0.2, lr=0.001):

    model = Sequential()

    model.add(LSTM(units, return_sequences=True, input_shape=(lookback, len(features))))
    model.add(Dropout(dropout))

    model.add(LSTM(units))
    model.add(Dropout(dropout))

    model.add(Dense(32))
    model.add(Dense(2))

    optimizer = Adam(learning_rate=lr)

    model.compile(optimizer=optimizer, loss='mean_squared_error')

    return model

In [44]:
# param_grid = {
#     "units": [32, 64, 128],
#     "dropout": [0.1, 0.2, 0.3],
#     "lr": [0.001, 0.0005],
#     "batch_size": [16, 32]
# }
param_grid = {
    "units": [64, 128],
    "dropout": [0.2, 0.3],
    "lr": [0.001],
    "batch_size": [32]
}

In [45]:
import itertools

results = []

for units, dropout, lr, batch_size in itertools.product(
    param_grid['units'],
    param_grid['dropout'],
    param_grid['lr'],
    param_grid['batch_size']
):

    print(f"Testing: units={units}, dropout={dropout}, lr={lr}, batch={batch_size}")

    model = build_model(units, dropout, lr)

    history = model.fit(
        X_train, y_train,
        validation_data=(X_test, y_test),
        epochs=10,   # keep it small for tuning
        batch_size=batch_size,
        verbose=0
    )

    val_loss = min(history.history['val_loss'])

    results.append({
        "units": units,
        "dropout": dropout,
        "lr": lr,
        "batch_size": batch_size,
        "val_loss": val_loss
    })

Testing: units=64, dropout=0.2, lr=0.001, batch=32
Testing: units=64, dropout=0.3, lr=0.001, batch=32
Testing: units=128, dropout=0.2, lr=0.001, batch=32
Testing: units=128, dropout=0.3, lr=0.001, batch=32


In [46]:
results_df = pd.DataFrame(results)

best = results_df.sort_values('val_loss').iloc[0]

print("BEST PARAMS:")
print(best)

BEST PARAMS:
units         128.000000
dropout         0.200000
lr              0.001000
batch_size     32.000000
val_loss        0.009625
Name: 2, dtype: float64


In [47]:
best_model = build_model(
    units=int(best['units']),
    dropout=float(best['dropout']),
    lr=float(best['lr'])
)

best_model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=30,
    batch_size=int(best['batch_size'])
)

Epoch 1/30
18/18 [==============================] - 11s 199ms/step - loss: 0.0164 - val_loss: 0.0139
Epoch 2/30
18/18 [==============================] - 2s 117ms/step - loss: 0.0036 - val_loss: 0.0112
Epoch 3/30
18/18 [==============================] - 2s 109ms/step - loss: 0.0031 - val_loss: 0.0119
Epoch 4/30
18/18 [==============================] - 2s 115ms/step - loss: 0.0026 - val_loss: 0.0106
Epoch 5/30
18/18 [==============================] - 2s 125ms/step - loss: 0.0022 - val_loss: 0.0111
Epoch 6/30
18/18 [==============================] - 2s 114ms/step - loss: 0.0017 - val_loss: 0.0102
Epoch 7/30
18/18 [==============================] - 2s 123ms/step - loss: 0.0017 - val_loss: 0.0097
Epoch 8/30
18/18 [==============================] - 2s 132ms/step - loss: 0.0019 - val_loss: 0.0100
Epoch 9/30
18/18 [==============================] - 2s 108ms/step - loss: 0.0016 - val_loss: 0.0099
Epoch 10/30
18/18 [==============================] - 2s 109ms/step - loss: 0.0017 - val_loss: 0.010